Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\1pasos_gru_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 7)
Dimensiones de Y: (43788, 1)


In [9]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 12, 7)
Las dimensiones de testX son:  (8801, 12, 7)
Las dimensiones de valX son:  (4336, 12, 7)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

49/49 - 20s - 411ms/step - ia: 0.2319 - loss: 1.6873 - mae: 1.0893 - rmse: 1.2946 - smape: 1.5843 - val_ia: 0.2422 - val_loss: 0.9272 - val_mae: 0.8456 - val_rmse: 0.9465 - val_smape: 1.6348

Epoch 2/128                                           

49/49 - 3s - 64ms/step - ia: 0.2242 - loss: 1.5035 - mae: 0.9991 - rmse: 1.2262 - smape: 1.5676 - val_ia: 0.2502 - val_loss: 0.7647 - val_mae: 0.7507 - val_rmse: 0.8556 - val_smape: 1.6542

Epoch 3/128                                           

49/49 - 1s - 21ms/step - ia: 0.2316 - loss: 1.4119 - mae: 0.9451 - rmse: 1.1919 - smape: 1.5539 - val_ia: 0.2635 - val_loss: 0.6696 - val_mae: 0.6858 - val_rmse: 0.7950 - val_smape: 1.6896

Epoch 4/128                                           

49/49 - 1s - 21ms/step - ia: 0.2221 - loss: 1.3855 - mae: 0.9213 - rmse: 1.1710 - smape: 1.5505 - val_ia: 0.2637 - val_loss: 0.6211 - val_mae: 0.6476 - val_rmse: 0.7606 - val_smape: 1.7308

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

385/385 - 29s - 76ms/step - ia: 0.8129 - loss: 0.1540 - mae: 0.2553 - rmse: 0.3552 - smape: 0.5533 - val_ia: 0.5396 - val_loss: 0.0736 - val_mae: 0.1771 - val_rmse: 0.2278 - val_smape: 0.5382

Epoch 2/128                                                                       

385/385 - 7s - 19ms/step - ia: 0.8798 - loss: 0.0735 - mae: 0.1717 - rmse: 0.2490 - smape: 0.3996 - val_ia: 0.6300 - val_loss: 0.0582 - val_mae: 0.1402 - val_rmse: 0.1916 - val_smape: 0.4579

Epoch 3/128                                                                       

385/385 - 12s - 31ms/step - ia: 0.8930 - loss: 0.0627 - mae: 0.1544 - rmse: 0.2276 - smape: 0.3616 - val_ia: 0.6443 - val_loss: 0.0547 - val_mae: 0.1353 - val_rmse: 0.1817 - val_smape: 0.4377

Epoch 4/128                                                                       

385/385 - 10s - 25ms/step - ia: 0.8963 - loss: 0.0593 - mae: 0.1482 - rmse: 0.2215 - s

...

Epoch 1/128                                                                       

385/385 - 11s - 28ms/step - ia: 0.2816 - loss: 1.4022 - mae: 0.8912 - rmse: 1.1481 - smape: 1.4474 - val_ia: 0.2496 - val_loss: 0.4922 - val_mae: 0.5339 - val_rmse: 0.5914 - val_smape: 1.4704

Epoch 2/128                                                                       

385/385 - 4s - 11ms/step - ia: 0.3168 - loss: 1.1414 - mae: 0.8036 - rmse: 1.0377 - smape: 1.4027 - val_ia: 0.2650 - val_loss: 0.4022 - val_mae: 0.4793 - val_rmse: 0.5337 - val_smape: 1.3141

Epoch 3/128                                                                       

385/385 - 6s - 15ms/step - ia: 0.4545 - loss: 0.7883 - mae: 0.6571 - rmse: 0.8543 - smape: 1.1945 - val_ia: 0.3254 - val_loss: 0.2335 - val_mae: 0.3530 - val_rmse: 0.4083 - val_smape: 0.9340

Epoch 4/128                                                                       

385/385 - 4s - 11ms/step - ia: 0.6266 - loss: 0.4715 - mae: 0.5078 - rmse: 0.6619 - sma

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                        

770/770 - 17s - 22ms/step - ia: 0.2961 - loss: 1.3370 - mae: 0.9271 - rmse: 1.1104 - smape: 1.5720 - val_ia: 0.1862 - val_loss: 0.5670 - val_mae: 0.6268 - val_rmse: 0.6596 - val_smape: 1.6876

Epoch 2/128                                                                        

770/770 - 5s - 6ms/step - ia: 0.2973 - loss: 1.3267 - mae: 0.9286 - rmse: 1.1055 - smape: 1.5792 - val_ia: 0.1867 - val_loss: 0.5636 - val_mae: 0.6242 - val_rmse: 0.6571 - val_smape: 1.6903

Epoch 3/128                                                                        

770/770 - 4s - 6ms/step - ia: 0.2931 - loss: 1.3464 - mae: 0.9321 - rmse: 1.1149 - smape: 1.5833 - val_ia: 0.1872 - val_loss: 0.5605 - val_mae: 0.6218 - val_rmse: 0.6547 - val_smape: 1.6928

Epoch 4/128                                                                        

770/770 - 4s - 6ms/step - ia: 0.2962 - loss: 1.3390 - mae: 0.9230 - rmse: 1.1102 - sm

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

49/49 - 7s - 151ms/step - ia: 0.2546 - loss: 1.1731 - mae: 0.8119 - rmse: 1.0791 - smape: 1.4724 - val_ia: 0.3028 - val_loss: 0.4617 - val_mae: 0.5213 - val_rmse: 0.6367 - val_smape: 1.4862

Epoch 2/128                                                                          

49/49 - 1s - 20ms/step - ia: 0.3624 - loss: 0.9279 - mae: 0.7145 - rmse: 0.9512 - smape: 1.3211 - val_ia: 0.3786 - val_loss: 0.3512 - val_mae: 0.4456 - val_rmse: 0.5516 - val_smape: 1.2217

Epoch 3/128                                                                          

49/49 - 2s - 32ms/step - ia: 0.4858 - loss: 0.6971 - mae: 0.6114 - rmse: 0.8330 - smape: 1.1432 - val_ia: 0.4704 - val_loss: 0.2626 - val_mae: 0.3812 - val_rmse: 0.4794 - val_smape: 1.0444

Epoch 4/128                                                                          

49/49 - 1s - 25ms/step - ia: 0.5965 - loss: 0.5036 - mae: 0.5219 - rmse: 0.7076 -

In [16]:
print(best)

{'activation': 3, 'batch': 4, 'dropout': 0.30000000000000004, 'layers': 1.0, 'learning_rate': 0.001217652235385441, 'units': 0}


In [17]:
# {'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4} 